# LLM03: Backpropagation and Autograd

## Lab Overview

This lab covers the **backward pass** of deep learning: how gradients are computed, how PyTorch's autograd engine works, and how to build a complete training loop. These concepts are essential for understanding how LLMs learn their parameters.

> **Prerequisites**: LLM02 (tensor operations, `nn.Linear`, forward propagation)

#### Recommended Hardware

AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)

#### Software Environment

OS: Ubuntu 24.04.3 LTS \
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=…). After installing AUP Learning Cloud, you will have a ROCm and PyTorch environment that is compatible with this notebook.

## Goals

By the end of this lab, you will:

- Understand forward propagation and backward propagation (chain rule)
- Know how PyTorch builds and uses **dynamic computation graphs**
- Manually verify gradients of a linear layer against autograd
- Use **hooks** to inspect gradient flow
- Master `zero_grad`, gradient accumulation, and training loop patterns
- Understand `detach()`, `no_grad()`, and `inference_mode()`
- Complete the A0 exercise: verify $dW$ and $dX$ for $Y = XW$

---


## 1. Environment Setup


In [41]:
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)

Using device: cuda
PyTorch version: 2.9.1+rocm7.2.0.git7e1940d4
GPU: AMD Radeon(TM) 8060S Graphics


## 2. Forward and Backward Propagation

### 2.1 The Big Picture

Training a neural network is an optimization problem: we want to find parameters $\theta$ that minimize a loss function over a dataset.

For a regression example with mean squared error (MSE) loss:

$$
J(\theta) = \frac{1}{n} \sum_{i=1}^{n} \left( y_i - f(x_i; \theta) \right)^2
$$

where:

- $x_i$ is the input
- $y_i$ is the target
- $f(x_i; \theta)$ is the model prediction
- $\theta$ denotes all learnable parameters

The training loop repeats three main steps:

1. **Forward pass**: compute predictions $\hat{Y} = f(X; \theta)$
2. **Compute loss**: measure the difference between predictions $\hat{Y}$ and targets $Y$
3. **Backward pass**: compute gradients $\frac{\partial J}{\partial \theta}$, then update parameters with an optimizer

A typical gradient-based update looks like:

$$
\theta \leftarrow \theta - \lambda \nabla_\theta J
$$

where $\lambda$ is the learning rate.

---

### 2.2 Chain Rule for Composite Functions

Deep learning models are **composite functions**.  
For example, a network with three stages can be written as:

$$
h_1 = f_1(x), \qquad h_2 = f_2(h_1), \qquad y = f_3(h_2)
$$

So the final output depends on earlier intermediate results step by step.

To compute how the loss changes with respect to an early parameter set $\theta_1$, we apply the **chain rule**:

$$
\frac{\partial J}{\partial \theta_1}
=
\frac{\partial J}{\partial y}
\cdot
\frac{\partial y}{\partial h_2}
\cdot
\frac{\partial h_2}{\partial h_1}
\cdot
\frac{\partial h_1}{\partial \theta_1}
$$

This is the key idea behind **backpropagation**:

- gradients are propagated from the output back toward earlier layers
- each layer receives an upstream gradient from the layer after it
- local derivatives are multiplied together by the chain rule

PyTorch's autograd system performs this process automatically by building a computation graph during the forward pass and traversing it in reverse during the backward pass.


In [42]:
# Demonstrate forward + backward on a simple 3-layer network

# Build a tiny 3-layer network: f3(f2(f1(x)))
f1 = nn.Linear(4, 8)
f2 = nn.Linear(8, 8)
f3 = nn.Linear(8, 2)

x = torch.randn(1, 4)
target = torch.tensor([[1.0, 0.0]])

# ----- Forward pass -----
h1 = F.relu(f1(x))  # layer 1 + activation
h2 = F.relu(f2(h1))  # layer 2 + activation
y_pred = f3(h2)  # layer 3 (output)

loss = F.mse_loss(y_pred, target)
print("Forward pass:")
print(f"  x      shape: {x.shape}")
print(f"  h1     shape: {h1.shape}")
print(f"  h2     shape: {h2.shape}")
print(f"  y_pred shape: {y_pred.shape}")
print(f"  loss = {loss.item():.6f}")

# ----- Backward pass -----
loss.backward()

print("\nBackward pass (gradient norms):")
for name, layer in [("f1", f1), ("f2", f2), ("f3", f3)]:
    grad_norm = layer.weight.grad.norm().item()
    print(f"  {name}.weight.grad norm: {grad_norm:.6f}")

Forward pass:
  x      shape: torch.Size([1, 4])
  h1     shape: torch.Size([1, 8])
  h2     shape: torch.Size([1, 8])
  y_pred shape: torch.Size([1, 2])
  loss = 0.250565

Backward pass (gradient norms):
  f1.weight.grad norm: 0.153468
  f2.weight.grad norm: 0.106831
  f3.weight.grad norm: 0.171273


## 3. Computation Graph

PyTorch builds a **dynamic computation graph** (usually a DAG) during the forward pass:

- **Nodes**: variables (tensors) and operations
- **Edges**: data flow between operations
- **Forward**: follow edges forward to compute output
- **Backward**: follow edges backward to compute gradients

Every tensor produced by an operation has a `grad_fn` attribute that records _how_ it was created.

### Key Concepts

- **Leaf tensors**: created directly by the user (e.g., parameters, inputs with `requires_grad=True`). Their `grad_fn` is `None` and gradients accumulate in `.grad`.
- **Non-leaf tensors**: produced by operations (e.g., `z = x + y`). They have a `grad_fn` (a `Function` object) describing how to compute gradients.
- By default, PyTorch stores `.grad` only for leaf tensors.  
  If you want gradients for a non-leaf tensor, call `retain_grad()` on it first.


In [43]:
# Explore the computation graph
print("\n=== One backward path (out → ... → x) ===")
x = torch.ones(2, 2, requires_grad=True)
y = x + 2
z = y * y * 3
out = z.mean()

print("=== Computation Graph Nodes ===")
print(f"x.grad_fn   = {x.grad_fn}          (leaf tensor, None)")
print(f"y.grad_fn   = {y.grad_fn}   (y = x + 2)")
print(f"z.grad_fn   = {z.grad_fn}   (z = y * y * 3)")
print(f"out.grad_fn = {out.grad_fn}  (out = z.mean())")

print("\n=== Leaf / Non-leaf ===")
print(f"x is leaf: {x.is_leaf}")
print(f"y is leaf: {y.is_leaf}")
print(f"z is leaf: {z.is_leaf}")

# Walk the graph backward from out
print("\n=== Graph walk (out → ... → x) ===")
node = out.grad_fn
depth = 0
while node is not None:
    indent = "  " * depth
    print(f"{indent}{type(node).__name__}")
    children = node.next_functions
    if children:
        # Follow the first input branch
        node = children[0][0]
    else:
        node = None
    depth += 1
print(f"{'  ' * depth}(leaf tensor x)")


=== One backward path (out → ... → x) ===
=== Computation Graph Nodes ===
x.grad_fn   = None          (leaf tensor, None)
y.grad_fn   = <AddBackward0 object at 0x769a64f4df30>   (y = x + 2)
z.grad_fn   = <MulBackward0 object at 0x769a64f4df30>   (z = y * y * 3)
out.grad_fn = <MeanBackward0 object at 0x769a64f4df30>  (out = z.mean())

=== Leaf / Non-leaf ===
x is leaf: True
y is leaf: False
z is leaf: False

=== Graph walk (out → ... → x) ===
MeanBackward0
  MulBackward0
    MulBackward0
      AddBackward0
        AccumulateGrad
          (leaf tensor x)


### 3.1 Common Backward Node Names

| Node Name        | Corresponds to           | Notes                                               |
| ---------------- | ------------------------ | --------------------------------------------------- |
| `AddmmBackward`  | `nn.Linear` / `F.linear` | $Y = XW^T + b$; gradients: $dX = GW$, $dW = G^T X$  |
| `TBackward`      | `.t()` / `transpose`     | Transpose the gradient back to original layout      |
| `AccumulateGrad` | Leaf tensor              | Not an operator — accumulates gradient into `.grad` |
| `ReluBackward`   | `F.relu`                 | Passes gradient where input > 0, zeros elsewhere    |
| `MulBackward`    | `*` (element-wise)       | Product rule: $d(ab) = a\,db + b\,da$               |
| `MeanBackward`   | `.mean()`                | Divides upstream gradient by number of elements     |

> Note: exact node names may vary slightly by PyTorch version/device, e.g. `AddmmBackward0` instead of `AddmmBackward`.


In [44]:
# Show grad_fn for a nn.Linear forward pass

linear = nn.Linear(4, 3)
x = torch.randn(2, 4)
y = linear(x)
loss = y.sum()

print("=== nn.Linear computation graph ===")
print(f"y.grad_fn       = {y.grad_fn}")


# Walk all children
def print_graph(node, indent=0):
    if node is None:
        return
    prefix = "  " * indent
    print(f"{prefix}{type(node).__name__}")
    for child, _ in node.next_functions:
        if child is not None:
            print_graph(child, indent + 1)
        else:
            print(f"{'  ' * (indent + 1)}(leaf)")


print("\n=== Full graph from loss ===")
print_graph(loss.grad_fn)

=== nn.Linear computation graph ===
y.grad_fn       = <AddmmBackward0 object at 0x769a64fae860>

=== Full graph from loss ===
SumBackward0
  AddmmBackward0
    AccumulateGrad
    (leaf)
    TBackward0
      AccumulateGrad


## 4. Linear Layer Backward Pass — Manual vs. Autograd

For a linear layer, PyTorch computes the forward pass as:

$$
Z = XW^T + b
$$

where:

- $X \in \mathbb{R}^{B \times D_{\text{in}}}$ is the input
- $W \in \mathbb{R}^{D_{\text{out}} \times D_{\text{in}}}$ is the weight matrix
- $b \in \mathbb{R}^{D_{\text{out}}}$ is the bias
- $Z \in \mathbb{R}^{B \times D_{\text{out}}}$ is the output

Suppose the upstream gradient from later computations is:

$$
G = \frac{\partial \text{loss}}{\partial Z}
\quad\text{with shape } B \times D_{\text{out}}
$$

Then the gradients are:

$$
\frac{\partial \text{loss}}{\partial W} = G^T X
$$

$$
\frac{\partial \text{loss}}{\partial b} = \sum_{\text{batch}} G
$$

$$
\frac{\partial \text{loss}}{\partial X} = GW
$$

Intuition:

- **Weight gradient** tells us how each output dimension depends on each input dimension
- **Bias gradient** is the sum of upstream gradients over the batch
- **Input gradient** tells us how the loss flows back to the previous layer

Below, we compute these gradients manually and compare them with PyTorch autograd.


In [45]:
# Manual gradient computation vs. autograd for nn.Linear
import torch
import torch.nn as nn

torch.manual_seed(0)

batch, in_dim, out_dim = 4, 3, 2

# Input X: shape (batch, in_dim)
x = torch.randn(batch, in_dim, requires_grad=True)

# nn.Linear stores weight as W with shape (out_dim, in_dim)
linear = nn.Linear(in_dim, out_dim)

# Forward: Z = XW^T + b
z = linear(x)

# Use a simple scalar loss so backward() can be called directly
loss = (z**2).sum()

# -----------------------------
# Autograd gradients
# -----------------------------
loss.backward()

grad_w_auto = linear.weight.grad.clone()  # shape: (out_dim, in_dim)
grad_b_auto = linear.bias.grad.clone()  # shape: (out_dim,)
grad_x_auto = x.grad.clone()  # shape: (batch, in_dim)

# -----------------------------
# Manual gradients
# -----------------------------
# Since loss = sum(z^2), we have:
# d(loss)/dZ = 2Z
G = 2 * z.detach()  # shape: (batch, out_dim)

# For Z = XW^T + b:
# d(loss)/dW = G^T X
grad_w_manual = G.T @ x.detach()  # shape: (out_dim, in_dim)

# d(loss)/db = sum of G over the batch dimension
grad_b_manual = G.sum(dim=0)  # shape: (out_dim,)

# d(loss)/dX = GW
# Note: W is stored as (out_dim, in_dim), so G @ W gives (batch, in_dim)
grad_x_manual = G @ linear.weight.detach()  # shape: (batch, in_dim)

# -----------------------------
# Verify results
# -----------------------------
print("=== Manual vs. Autograd Gradient Verification ===")
print(f"dW match: {torch.allclose(grad_w_auto, grad_w_manual, atol=1e-6)}")
print(f"db match: {torch.allclose(grad_b_auto, grad_b_manual, atol=1e-6)}")
print(f"dX match: {torch.allclose(grad_x_auto, grad_x_manual, atol=1e-6)}")

print("\nShapes:")
print(f"  dW: {grad_w_auto.shape}  (out_dim, in_dim)")
print(f"  db: {grad_b_auto.shape}  (out_dim,)")
print(f"  dX: {grad_x_auto.shape}  (batch, in_dim)")

print("\nSample values:")
print("Autograd dW:\n", grad_w_auto)
print("Manual   dW:\n", grad_w_manual)

print("\nAutograd db:\n", grad_b_auto)
print("Manual   db:\n", grad_b_manual)

print("\nAutograd dX:\n", grad_x_auto)
print("Manual   dX:\n", grad_x_manual)

=== Manual vs. Autograd Gradient Verification ===
dW match: True
db match: True
dX match: True

Shapes:
  dW: torch.Size([2, 3])  (out_dim, in_dim)
  db: torch.Size([2])  (out_dim,)
  dX: torch.Size([4, 3])  (batch, in_dim)

Sample values:
Autograd dW:
 tensor([[-6.2938,  2.3969, 10.4531],
        [ 0.5425, -2.7183, -2.4382]])
Manual   dW:
 tensor([[-6.2938,  2.3969, 10.4531],
        [ 0.5425, -2.7183, -2.4382]])

Autograd db:
 tensor([-6.6339,  3.7601])
Manual   db:
 tensor([-6.6339,  3.7601])

Autograd dX:
 tensor([[ 0.1783, -0.2581, -1.6237],
        [-0.6493, -0.7078, -1.3562],
        [ 0.0201, -0.1324, -0.6385],
        [-0.9456, -0.6737, -0.3993]])
Manual   dX:
 tensor([[ 0.1783, -0.2581, -1.6237],
        [-0.6493, -0.7078, -1.3562],
        [ 0.0201, -0.1324, -0.6385],
        [-0.9456, -0.6737, -0.3993]])


## 5. Scalar vs. Non-scalar Backward

- **Scalar output**: `loss.backward()` works directly — the starting gradient is implicitly `1`.
- **Non-scalar output**: You must provide `gradient=v` where `v` has the same shape as the output. This represents the upstream gradient (i.e., a **vector-Jacobian product**, VJP).

For example, `y.sum().backward()` is equivalent to `y.backward(torch.ones_like(y))`, because if

$$
L = \sum_i y_i
$$

then

$$
\frac{\partial L}{\partial y} = [1,1,\dots,1].
$$

In practice, we usually reduce to a scalar loss before calling `.backward()`. But understanding VJP is important for advanced use cases.


In [46]:
# Scalar backward
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * x  # element-wise square
loss = y.sum()  # scalar
loss.backward()
print(f"Scalar backward: x.grad = {x.grad}")  # d(x^2)/dx = 2x

# Non-scalar backward (VJP)
x2 = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y2 = x2 * x2  # non-scalar output

# This would fail:
try:
    y2.backward()
except RuntimeError as e:
    print(f"\nNon-scalar backward without gradient fails: {e}")

# Provide upstream gradient vector v (same shape as y2)
x3 = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y3 = x3 * x3
v = torch.tensor([1.0, 1.0, 1.0])  # equivalent to .sum().backward()
y3.backward(gradient=v)
print(f"VJP backward:    x.grad = {x3.grad}")

Scalar backward: x.grad = tensor([2., 4., 6.])

Non-scalar backward without gradient fails: grad can be implicitly created only for scalar outputs
VJP backward:    x.grad = tensor([2., 4., 6.])


## 6. Hooks: Inspecting Gradient Flow

**Hooks** are mainly useful for inspecting gradient flow during the backward pass.
They can also modify gradients, but in practice they are most often used for debugging and analysis.

- `tensor.register_hook(fn)` — called every time a gradient is computed for that tensor
- `module.register_full_backward_hook(fn)` — called after a module's backward

This is invaluable for debugging vanishing/exploding gradients in deep networks.


In [47]:
# Tensor-level hook: log gradient stats
grad_log = []


def log_grad(name):
    def hook(grad):
        grad_log.append(
            {"name": name, "mean": grad.mean().item(), "norm": grad.norm().item(), "shape": tuple(grad.shape)}
        )

    return hook


# Build a small network
net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 8),
    nn.ReLU(),
    nn.Linear(8, 2),
)

# Register hooks on the output of each linear layer
x = torch.randn(2, 4)
target = torch.randn(2, 2)

# Forward with hooks
h = x
for i, layer in enumerate(net):
    h = layer(h)
    if isinstance(layer, nn.Linear):
        h.register_hook(log_grad(f"after_linear_{i}"))

loss = F.mse_loss(h, target)
loss.backward()

print("=== Gradient Flow (via hooks) ===")
for entry in reversed(grad_log):  # reverse = forward order
    print(f"  {entry['name']:>20s}:  mean={entry['mean']:+.6f}  norm={entry['norm']:.6f}  shape={entry['shape']}")

=== Gradient Flow (via hooks) ===
        after_linear_0:  mean=+0.005154  norm=0.273062  shape=(2, 8)
        after_linear_2:  mean=+0.029822  norm=0.586236  shape=(2, 8)
        after_linear_4:  mean=-0.229233  norm=1.309122  shape=(2, 2)


In [48]:
# Module-level backward hook: inspect gradient at each layer
layer_grads = {}


def backward_hook(name):
    def hook(module, grad_input, grad_output):
        layer_grads[name] = {
            "grad_output_norm": grad_output[0].norm().item() if grad_output[0] is not None else None,
            "grad_input_norm": grad_input[0].norm().item() if grad_input[0] is not None else None,
        }

    return hook


# Register on each Linear layer
handles = []
for name, module in net.named_modules():
    if isinstance(module, nn.Linear):
        h = module.register_full_backward_hook(backward_hook(name))
        handles.append(h)

# Fresh forward + backward
net.zero_grad()
y = net(x)
loss = F.mse_loss(y, target)
loss.backward()

print("=== Module Backward Hooks ===")
for name, info in layer_grads.items():
    go = f"{info['grad_output_norm']:.6f}" if info["grad_output_norm"] is not None else "None"
    gi = f"{info['grad_input_norm']:.6f}" if info["grad_input_norm"] is not None else "None"
    print(f"  {name}: grad_output_norm={go}, grad_input_norm={gi}")

# Clean up hooks
for h in handles:
    h.remove()

=== Module Backward Hooks ===
  4: grad_output_norm=1.309122, grad_input_norm=0.754206
  2: grad_output_norm=0.586236, grad_input_norm=0.350712
  0: grad_output_norm=0.273062, grad_input_norm=None


## 7. Gradient Accumulation and Zeroing

**Critical**: PyTorch **accumulates** gradients by default. Calling `loss.backward()` multiple times **adds** to `.grad` instead of replacing it.

The standard training loop pattern:

```python
optimizer.zero_grad(set_to_none=True)  # clear old gradients
loss.backward()                         # compute new gradients
optimizer.step()                        # update parameters
```

`set_to_none=True` is faster and uses less memory than zeroing. With `set_to_none=True`, gradients become `None` instead of explicit zero tensors. This is expected and usually preferred for performance.


In [49]:
# Demonstrate gradient accumulation

linear = nn.Linear(3, 1)
x = torch.randn(2, 3)

# First backward
y1 = linear(x).sum()
y1.backward()
print(f"After 1st backward: weight.grad = {linear.weight.grad}")

# Second backward WITHOUT zero_grad — gradients ACCUMULATE
y2 = linear(x).sum()
y2.backward()
print(f"After 2nd backward: weight.grad = {linear.weight.grad}")
print("(Notice: 2nd grad ≈ 2 × 1st grad — they accumulated!)")

# Correct pattern: zero_grad first
linear.zero_grad(set_to_none=True)
print(f"\nAfter zero_grad:    weight.grad = {linear.weight.grad}")

y3 = linear(x).sum()
y3.backward()
print(f"After clean backward: weight.grad = {linear.weight.grad}")

After 1st backward: weight.grad = tensor([[1.6284, 0.1084, 0.2258]])
After 2nd backward: weight.grad = tensor([[3.2567, 0.2168, 0.4516]])
(Notice: 2nd grad ≈ 2 × 1st grad — they accumulated!)

After zero_grad:    weight.grad = None
After clean backward: weight.grad = tensor([[1.6284, 0.1084, 0.2258]])


In [50]:
# Intentional gradient accumulation: simulate larger batch
# Useful when GPU memory is limited

model = nn.Linear(4, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# Simulate: accumulate gradients over 4 mini-batches of size 8
# Effective batch size = 4 * 8 = 32
accum_steps = 4
optimizer.zero_grad(set_to_none=True)

for step in range(accum_steps):
    x = torch.randn(8, 4)
    target = torch.randn(8, 2)
    output = model(x)
    loss = criterion(output, target) / accum_steps  # scale loss
    loss.backward()  # accumulate gradients
    print(f"  Step {step}: loss={loss.item():.4f}, weight.grad.norm={model.weight.grad.norm().item():.4f}")

optimizer.step()  # single update with accumulated gradients
print(f"\nParameter update applied (effective batch size = {accum_steps * 8})")

  Step 0: loss=0.2569, weight.grad.norm=0.3287
  Step 1: loss=0.1915, weight.grad.norm=0.6329
  Step 2: loss=0.4176, weight.grad.norm=0.6719
  Step 3: loss=0.2956, weight.grad.norm=0.8330

Parameter update applied (effective batch size = 32)


## 8. Complete Training Loop

Putting it all together: zero gradients → forward → loss → backward → update.

```
for epoch in range(num_epochs):
    optimizer.zero_grad()         # clear old gradients
    y_pred = model(x)             # forward
    loss = criterion(y_pred, y)   # compute loss
    loss.backward()               # backward (compute gradients)
    optimizer.step()              # update parameters
```


In [51]:
# Full training loop: learn y = 2x + 1
torch.manual_seed(42)

# Generate data
X_train = torch.rand(100, 1) * 10  # x in [0, 10]
Y_train = 2 * X_train + 1 + torch.randn(100, 1) * 0.5  # add noise

# Model
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

print(f"Before training: w={model.weight.item():.4f}, b={model.bias.item():.4f}")

# Training loop
losses = []
for epoch in range(200):
    optimizer.zero_grad(set_to_none=True)  # zero gradients
    y_pred = model(X_train)  # forward
    loss = criterion(y_pred, Y_train)  # loss

    loss.backward()  # backward
    optimizer.step()  # update

    losses.append(loss.item())
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1:>3d}: loss={loss.item():.4f}, w={model.weight.item():.4f}, b={model.bias.item():.4f}")

print(f"\nLearned: y = {model.weight.item():.4f}x + {model.bias.item():.4f}")
print("Target:  y = 2.0000x + 1.0000")
print(f"Loss reduced: {losses[0]:.4f} → {losses[-1]:.4f}")

Before training: w=0.1447, b=-0.2590
Epoch  50: loss=0.3440, w=2.1024, b=0.2650
Epoch 100: loss=0.2762, w=2.0757, b=0.4461
Epoch 150: loss=0.2342, w=2.0546, b=0.5886
Epoch 200: loss=0.2082, w=2.0381, b=0.7007

Learned: y = 2.0381x + 0.7007
Target:  y = 2.0000x + 1.0000
Loss reduced: 146.1965 → 0.2082


## 9. Detach, no_grad, and inference_mode

PyTorch provides several ways to **stop gradient tracking**:

### 9.1 `detach()`

Detaches a tensor from the computation graph. The result shares storage with the original but has no `grad_fn`.

**Use cases**: truncate gradient flow (e.g., frozen teacher in distillation), cache intermediate results.

### 9.2 `torch.no_grad()`

Context manager that disables gradient computation. No `grad_fn` is recorded for operations inside.

**Use cases**: evaluation/validation, manual parameter updates (e.g., EMA), metrics computation.

### 9.3 `torch.inference_mode()`

Stricter than `no_grad`: also skips version counting and view tracking for extra speed.

**Use case**: pure inference / deployment.

| Feature          | `no_grad`       | `inference_mode`    |
| ---------------- | --------------- | ------------------- |
| Builds graph     | No              | No                  |
| Version counting | Yes             | Skipped             |
| Memory / Speed   | Good            | Better              |
| Safe in training | Yes (local use) | No (inference only) |

`inference_mode()` is best for pure inference. Tensors created inside it are more restricted and are not meant to re-enter normal autograd workflows later.

`torch.no_grad()` is more flexible: it can also be useful when part of a model is kept frozen while other parts are fine-tuned. In practice, freezing is usually done by setting `requires_grad=False` on the frozen parameters, while `no_grad()` disables gradient tracking for a specific block of computation.


In [52]:
# detach()
x = torch.tensor([2.0], requires_grad=True)
y = x * 3
z = y.detach() * 2  # z is detached — no gradient flows through here

print("=== detach() ===")
print(f"y.grad_fn: {y.grad_fn}")
print(f"z.grad_fn: {z.grad_fn}  (None — detached!)")
print(f"y requires_grad: {y.requires_grad}")
print(f"z requires_grad: {z.requires_grad}")

# Gradient only flows through y, not through the detached z branch
loss = y.sum() + z.sum()
loss.backward()
print(f"x.grad = {x.grad}  (only from y branch: d(3x)/dx = 3, z branch detached)")

=== detach() ===
y.grad_fn: <MulBackward0 object at 0x769a64f13bb0>
z.grad_fn: None  (None — detached!)
y requires_grad: True
z requires_grad: False
x.grad = tensor([3.])  (only from y branch: d(3x)/dx = 3, z branch detached)


In [53]:
# no_grad() vs inference_mode()

model = nn.Linear(4, 2)
x = torch.randn(2, 4)

# Normal forward — has grad_fn
y_normal = model(x)
print(f"Normal:         y.grad_fn = {y_normal.grad_fn}")

# no_grad — no grad_fn
with torch.no_grad():
    y_nograd = model(x)
    print(f"no_grad:        y.grad_fn = {y_nograd.grad_fn}")

# inference_mode — no grad_fn, extra optimizations
with torch.inference_mode():
    y_infer = model(x)
    print(f"inference_mode: y.grad_fn = {y_infer.grad_fn}")

Normal:         y.grad_fn = <AddmmBackward0 object at 0x769a6ff22cb0>
no_grad:        y.grad_fn = None
inference_mode: y.grad_fn = None


In [54]:
# Practical pattern: evaluation loop

model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2),
)

# During training
model.train()
x = torch.randn(4, 4)
y_train = model(x)
print(f"Training mode: model.training={model.training}, has grad_fn={y_train.grad_fn is not None}")

# During evaluation (always combine model.eval() with no_grad/inference_mode)
model.eval()
with torch.inference_mode():
    y_eval = model(x)
    print(f"Eval mode:     model.training={model.training}, has grad_fn={y_eval.grad_fn is not None}")

Training mode: model.training=True, has grad_fn=True
Eval mode:     model.training=False, has grad_fn=False


### Evaluation vs training

- **`model.eval()`** changes the layer behavior.
- **`torch.no_grad()` / `inference_mode()`** disables gradient tracking.
- They solve different problems, so evaluation usually needs both.


## 10. Higher-order Gradients

By default, `backward()` frees the computation graph after one pass. For higher-order gradients:

- `create_graph=True` — build a graph of the backward pass itself (needed for 2nd-order derivatives)
- `retain_graph=True` — keep the graph for multiple backward passes


In [55]:
# Second-order gradient: d²(x³)/dx² = 6x

x = torch.tensor([2.0], requires_grad=True)
y = x**3  # y = x³

# First derivative: dy/dx = 3x² = 12.0
dy_dx = torch.autograd.grad(y, x, create_graph=True)[0]
print(f"dy/dx = {dy_dx.item():.1f}  (expected: 3 * 2² = 12.0)")

# Second derivative: d²y/dx² = 6x = 12.0
d2y_dx2 = torch.autograd.grad(dy_dx, x)[0]
print(f"d²y/dx² = {d2y_dx2.item():.1f}  (expected: 6 * 2 = 12.0)")

dy/dx = 12.0  (expected: 3 * 2² = 12.0)
d²y/dx² = 12.0  (expected: 6 * 2 = 12.0)


In [56]:
# retain_graph: multiple backward passes on the same graph

x = torch.tensor([3.0], requires_grad=True)
y = x**2

# First backward — normally this would free the graph
y.backward(retain_graph=True)  # keep graph alive
print(f"1st backward: x.grad = {x.grad.item():.1f}")

# Second backward on the SAME graph
x.grad = None  # reset manually
y.backward(retain_graph=True)
print(f"2nd backward: x.grad = {x.grad.item():.1f}")

# Without retain_graph, this would fail:
x.grad = None
y.backward()  # no retain_graph — graph is freed after this
print(f"3rd backward: x.grad = {x.grad.item():.1f}")

try:
    y.backward()  # graph already freed!
except RuntimeError as e:
    print(f"4th backward fails: {e}")

1st backward: x.grad = 6.0
2nd backward: x.grad = 6.0
3rd backward: x.grad = 6.0
4th backward fails: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.


## 11. A0 Exercise: Verify Gradients for $Y = XW$

### Problem Setup

- Linear transform: $Y = XW$
- Shapes: $X \in \mathbb{R}^{B \times H \times D}$, $W \in \mathbb{R}^{D \times E}$, $Y \in \mathbb{R}^{B \times H \times E}$
- Loss: $\text{Loss} = \sum Y^2$ (scalar, for easy backprop)

### Gradient Derivation

Upstream gradient: $G = \frac{\partial \text{Loss}}{\partial Y} = 2Y \in \mathbb{R}^{B \times H \times E}$

Flatten batch dimensions ($N = B \times H$):

- $dW = \text{reshape}(X, N, D)^T \cdot \text{reshape}(G, N, E) \in \mathbb{R}^{D \times E}$
- $dX = G \cdot W^T \in \mathbb{R}^{B \times H \times D}$

### Debugging Checklist

- Use `contiguous()` before `view()` if needed; `reshape()` is usually safer because it can handle non-contiguous tensors by making a copy when necessary.
- For bias: $db = G.\text{sum}(\text{dim}=(0,1))$
- Use `float64` and `torch.autograd.gradcheck` for numerical verification
- Use `tensor.register_hook` to inspect intermediate gradient stats


In [57]:
# A0: Manual gradient verification for Y = X @ W

torch.manual_seed(0)
B, H, D, E = 2, 3, 4, 5

X = torch.randn(B, H, D, dtype=torch.float64, requires_grad=True)
W = torch.randn(D, E, dtype=torch.float64, requires_grad=True)

# Forward
Y = X @ W  # (B, H, D) @ (D, E) → (B, H, E)
Loss = (Y**2).sum()  # scalar loss

# Autograd backward
Loss.backward()
grad_X_auto = X.grad.clone()
grad_W_auto = W.grad.clone()

# Manual backward
G = 2 * Y.detach()  # (B, H, E)
dW = X.detach().reshape(-1, D).T @ G.reshape(-1, E)  # (D, N) @ (N, E) → (D, E)
dX = torch.matmul(G, W.detach().T)  # (B, H, E) @ (E, D) → (B, H, D)

# Verify
print("=== A0: Y = X @ W Gradient Verification ===")
print(f"Shapes: X={tuple(X.shape)}, W={tuple(W.shape)}, Y={tuple(Y.shape)}")
print("")
print(f"dW match: {torch.allclose(grad_W_auto, dW, rtol=1e-6, atol=1e-6)}")
print(f"dX match: {torch.allclose(grad_X_auto, dX, rtol=1e-6, atol=1e-6)}")
print("")
print(f"grad_W_auto:\n{grad_W_auto}")
print(f"\ndW_manual:\n{dW}")
print(f"\nMax abs diff (dW): {(grad_W_auto - dW).abs().max().item():.2e}")
print(f"Max abs diff (dX): {(grad_X_auto - dX).abs().max().item():.2e}")

=== A0: Y = X @ W Gradient Verification ===
Shapes: X=(2, 3, 4), W=(4, 5), Y=(2, 3, 5)

dW match: True
dX match: True

grad_W_auto:
tensor([[ 29.8266,   2.9379,  17.5352, -11.4535,   9.8399],
        [ -4.9089,   5.3152,   7.3471,  -8.7742,  13.8296],
        [ 51.0075,  -6.7696,  16.8786,  -4.4732, -19.7853],
        [-52.6624,   9.8502, -11.2709,  -0.0819,  27.0332]],
       dtype=torch.float64)

dW_manual:
tensor([[ 29.8266,   2.9379,  17.5352, -11.4535,   9.8399],
        [ -4.9089,   5.3152,   7.3471,  -8.7742,  13.8296],
        [ 51.0075,  -6.7696,  16.8786,  -4.4732, -19.7853],
        [-52.6624,   9.8502, -11.2709,  -0.0819,  27.0332]],
       dtype=torch.float64)

Max abs diff (dW): 0.00e+00
Max abs diff (dX): 0.00e+00


In [58]:
# Bonus: Use torch.autograd.gradcheck for automatic numerical verification


def forward_fn(X, W):
    Y = X @ W
    return (Y**2).sum()


X_check = torch.randn(2, 3, 4, dtype=torch.float64, requires_grad=True)
W_check = torch.randn(4, 5, dtype=torch.float64, requires_grad=True)

# gradcheck compares autograd gradients with finite-difference approximation
result = torch.autograd.gradcheck(forward_fn, (X_check, W_check), eps=1e-6, atol=1e-4)
print(f"torch.autograd.gradcheck passed: {result}")

torch.autograd.gradcheck passed: True


## Conclusions

### Technical Concepts Learned

- **Forward & Backward Propagation**: The training loop — predict, compute loss, compute gradients, update parameters
- **Chain Rule**: Backpropagation is efficient chain-rule computation across composite functions
- **Computation Graph**: Dynamic DAG built during forward pass; `grad_fn` links record operations; common nodes include `AddmmBackward`, `TBackward`, `AccumulateGrad`
- **Manual Gradient Verification**: For $z = xW^T + b$: $dW = G^T x$, $db = \sum G$, $dx = GW$
- **Scalar vs. Non-scalar Backward**: Scalar outputs use implicit gradient=1; non-scalar require explicit `gradient=v` (VJP)
- **Hooks**: `register_hook` (tensor) and `register_full_backward_hook` (module) for gradient inspection
- **Gradient Accumulation**: `.grad` accumulates by default — always `zero_grad()` before backward; intentional accumulation simulates larger batches
- **Training Loop**: `zero_grad → forward → loss → backward → step`
- **Detach / no_grad / inference_mode**: Three levels of gradient disabling for different scenarios
- **Higher-order Gradients**: `create_graph=True` for 2nd derivatives; `retain_graph=True` for multiple backward passes

### What's Next

- **LLM04**: Tokenization, embedding lookup, and positional encoding (RoPE)
- **LLM05**: Normalization (RMSNorm) and FFN architectures (SwiGLU)

### Experiment Further

- Add a bias term to the A0 exercise and verify $db = G.\text{sum}(\text{dim}=(0,1))$
- Use hooks to plot gradient norms across layers for a 10-layer network — observe vanishing gradients
- Compare training convergence with different learning rates and optimizers (SGD vs. Adam)
- Try `torch.autograd.gradcheck` on your own custom function


---

Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
